In [1]:
import sys
import json
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings import FastEmbedEmbeddings

from src.config import (
    QDRANT_URL, 
    COLLECTION_NAME, 
    EMBEDDING_MODEL, 
    RETRIEVAL_TOP_K
)
from src.sidecar_manager import SidecarManager
from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget
from src.tag_assigner import TagAssigner
from src.tag_reranker import TagReranker

# Inizializzazione Vector Store con FastEmbed nativo
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
qdrant_client = QdrantClient(url=QDRANT_URL)

# Per il DataSet Pre-Taggato
sidecar_path = project_root / "data/processed/armstrong/sc_multi_pretag4.json"

# DataSet privo di Tag
#sidecar_path = project_root / "data/processed/armstrong/sidecar_nopretag3.json"

# Inizializzazione sidecar (runnare 1 sola volta altrimenti da il WARNING)
sidecar_mngr = SidecarManager(filepath=sidecar_path)

# Inizializzazione widget
widget = ChunkGraphWidget(sidecar=sidecar_mngr)

# Inizializzazione moduli gestinoe Tag
tag_assigner = TagAssigner()
tag_reranker = TagReranker(sidecar_manager=widget.sidecar, tag_assigner=tag_assigner)

print("!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita")

/tmp/ipykernel_771888/2193373322.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings
/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita


In [2]:
# Caricamento della query di test da eval_queries.json
queries_path = project_root / "data/queries/armstrong/eval_queries.json"
with open(queries_path, "r", encoding="utf-8") as f:
    benchmark_queries = json.load(f)

#Q_AMBIG_01, Q_AMBIG_02, Q_AMBIG_03, Q_SPEC_JAZZ_01, Q_SPEC_BIKE_01, Q_SPEC_ASTRO_01
matched = [q for q in benchmark_queries if q.get("id") == "Q_SPEC_ASTRO_01"]
query_obj = matched[0]
query_text = query_obj["query"]

print(f"?> ID Query: {query_obj['id']}")
print(f"  >>> Testo Query: '{query_text}'\n")

# Retrieval
query_vector = embeddings.embed_query(query_text)

ARMSTRONG_TOP_K = 5

response = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=ARMSTRONG_TOP_K,
    with_vectors=True,
    with_payload=True
)
raw_records = response.points

print(f"!>> Recuperati {len(raw_records)} record con relativi vettori di embedding")

?> ID Query: Q_SPEC_ASTRO_01
  >>> Testo Query: 'Quale ruolo ha avuto Armstrong nella missione Apollo 11 e quali frasi celebri ha pronunciato?'

!>> Recuperati 5 record con relativi vettori di embedding


In [3]:
# Grafo Retrieved
builder = KnowledgeGraphBuilder()
tag_overrides = sidecar_mngr.load_data().get("tag_overrides", {})

for idx, rec in enumerate(raw_records):
    payload = rec.payload or {}
    
    # Costruzione dell'identificativo unico del chunk
    meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload

    doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"

    raw_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
    chunk_idx = raw_idx if raw_idx is not None else idx_global

    chunk_id = payload.get("chunk_id") or meta.get("chunk_id") or f"{doc_id}_chunk_{chunk_idx}"
    
    # Estrazione testo
    text = payload.get("text", payload.get("page_content", ""))

    # Estrazione tag - da la priorità a quelli del sidecar, o quelli salvati su Qdrant (in teoria non presenti)
    base_tags = payload.get("user_tags") or meta.get("user_tags") or payload.get("tags") or meta.get("tags") or []
    chunk_sidecar_info = tag_overrides.get(chunk_id, {})
    tags = chunk_sidecar_info.get("user_tags", base_tags)
    
    builder.add_chunk_node(
        chunk_id=chunk_id,
        text=text,
        vector=rec.vector,
        tags=tags
    )

builder.auto_connect_nodes()

graph_data = builder.to_json_data()

print(f"!>> Grafo generato con successo:")
print(f"   >>> Nodi inseriti: {len(graph_data['nodes'])}")
print(f"   >>> Archi collegati: {len(graph_data['links'])}")

### Inizializzazione e rendering del Widget
widget.load_graph(graph_data)

widget

!>> Grafo generato con successo:
   >>> Nodi inseriti: 5
   >>> Archi collegati: 10


In [4]:
# Reranking
ARMSTRONG_TOP_N = 3

reranked_results = tag_reranker.rerank(
    query_text=query_text,
    retrieved_points=raw_records,
    top_n=ARMSTRONG_TOP_N
) 

print(f"!>> Reranking completato. Elaborati {len(reranked_results)} chunk candidati.\n")

if reranked_results:
    print(f"  >>> Tag assegnati alla Query: {reranked_results[0].get('query_tags', [])}")
    print("\n--- Dettaglio Punteggi Post-Reranking ---")
    for item in reranked_results:
        print(f"  > ID: {item['chunk_id']}")
        print(f"    Score Qdrant: {item['initial_score']} | Score Finale: {item['final_score']} (Moltiplicatore W_tag: {item['weight_factor']})")
        print(f"    Tag Chunk: {item['chunk_tags']} | Tag Coincidenti: {item['matched_tags']}\n")

DB>> tag: Jazz
DB>> expanded_tag:
  >>>Jazz: Il jazz è un genere musicale nato negli Stati Uniti alla fine del XIX secolo, caratterizzato da un mix di influenze africane, europee e latinoamericane, con strumenti come il saxofono, la chitarra e il piano, e sotto-categorie come il swing, il bebop e il free jazz, con artisti come Louis Armstrong, Charlie Parker e John Coltrane.
DB>> tag: Ciclismo
DB>> expanded_tag:
  >>>Ciclismo: l'attività sportiva che implica l'uso di biciclette, con sotto-categorie come il ciclismo su strada, il ciclismo su pista, il mountain bike e il ciclismo di strada, che richiedono strumenti e componenti chiave come le ruote, i freni, i pedali e le selle, e sono correlate con entità come le gare, le competizioni e le squadre, e termini tecnici come il "girone" e il "tempo".
DB>> tag: Astronautica
DB>> expanded_tag:
  >>>Astronautica: l'astronautica è un ramo della scienza e della tecnologia che si occupa dello sviluppo, della costruzione e dell'esplorazione delle 

In [5]:
# Grafo post-reranking
builder_reranked = KnowledgeGraphBuilder()

for item in reranked_results:
    payload = item.get("payload", {})
    text = payload.get("text", payload.get("page_content", ""))
    
    builder_reranked.add_chunk_node(
        chunk_id=item["chunk_id"],
        text=text,
        vector=item["vector"],
        tags=item["chunk_tags"],
    ) 

builder_reranked.auto_connect_nodes()
graph_data_reranked = builder_reranked.to_json_data()

# DEBUG print
print(f"!>> Grafo aggiornato post-reranking:")
print(f"   >>> Nodi: {len(graph_data_reranked['nodes'])}")
print(f"   >>> Archi: {len(graph_data_reranked['links'])}")

print("\n--- Analisi Impatto Reranking (Score & Moltiplicatori) ---")
for item in reranked_results:
    init_s = item["initial_score"]
    final_s = item["final_score"]
    delta_s = final_s - init_s
    w_factor = item["weight_factor"]
    matched = item["matched_tags"]
    
    print(
        f"Chunk: {item['chunk_id']:<22} | "
        f"Score: {init_s:.4f} -> {final_s:.4f} (Δ: {delta_s:+.4f}) | "
        f"W_tag: {w_factor:.2f}x | Match: {matched}"
    )
    
widget.load_graph(graph_data_reranked)
widget

!>> Grafo aggiornato post-reranking:
   >>> Nodi: 3
   >>> Archi: 3

--- Analisi Impatto Reranking (Score & Moltiplicatori) ---
Chunk: neil_armstrong_2       | Score: 0.7282 -> 0.8738 (Δ: +0.1456) | W_tag: 1.20x | Match: ['Astronautica']
Chunk: neil_armstrong_1       | Score: 0.7171 -> 0.8605 (Δ: +0.1434) | W_tag: 1.20x | Match: ['Astronautica']
Chunk: neil_armstrong_3       | Score: 0.7095 -> 0.8514 (Δ: +0.1419) | W_tag: 1.20x | Match: ['Astronautica']


In [6]:
# LLM pre e post rerank
from langchain_ollama import ChatOllama
from src.config import (
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    LLM_MODEL,
    OLLAMA_URL,
)

llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_URL,
    keep_alive=DEFAULT_KEEP_ALIVE,
    num_thread=DEFAULT_NUM_THREAD,
    temperature=0,
)

def format_raw_context(raw_records):
    context_blocks = []
    for idx, rec in enumerate(raw_records, start=1):
        payload = rec.payload or {}
        text = payload.get("text") or payload.get("page_content") or meta.get("text", "")
        score = getattr(rec, "score", 0.0)

        context_blocks.append(
            f"[CHUNK {idx} | Score Raw: {score:.4f}]\n{text.strip()}"
        )

    return "\n\n".join(context_blocks)

def format_reranked_context(reranked_results):
    context_blocks = []
    for idx, item in enumerate(reranked_results, start=1):
        payload = item.get("payload", {})
        text_content = (
            item.get("text") or payload.get("text") or payload.get("page_content", "")
        )
        chunk_id = item.get("chunk_id")
        score = item.get("final_score", 0.0)
    
        context_blocks.append(
            f"[CHUNK {idx} | Score Reranked: {score:.4f}]\n{text_content.strip()}"
        )
    
    return "\n\n".join(context_blocks)

context_pre = format_raw_context(raw_records)
context_post = format_reranked_context(reranked_results)

prompt_template = """Sei un assistente di ricerca specializzato sugli argomenti del contesto. Rispondi alla domanda seguente basandoti ESCLUSIVAMENTE sul contesto fornito. Se il contesto non contiene informazioni sufficienti o è parziale, evidenzialo chiaramente.

Domanda: {query_text}

Contesto recuperato:
{context}

Risposta:"""

# Invocazione del modello
response_pre = llm.invoke(
    prompt_template.format(query_text=query_text, context=context_pre)
)
response_post = llm.invoke(
    prompt_template.format(query_text=query_text, context=context_post)
)

# Stampa di Query e Risposta
print("\n>>> 1. RISPOSTA PRE-RERANKING (Vector Search Pure):\n")
print(response_pre.content.strip())

print("\n" + "-" * 80)

print("\n>>> 2. RISPOSTA POST-RERANKING (Reranked Context):\n")
print(response_post.content.strip())

print("\n" + "=" * 80)


>>> 1. RISPOSTA PRE-RERANKING (Vector Search Pure):

Ciao! Sono felice di aiutarti a rispondere alla tua domanda.

Neil Armstrong ha avuto un ruolo fondamentale nella missione Apollo 11 della NASA, che è stata la prima missione a portare un uomo sulla Luna. Armstrong è stato il comandante della missione e ha guidato il modulo lunare Eagle verso il mare della Tranquillità, dove ha effettuato la prima camminata sulla Luna.

La frase più celebre pronunciata da Armstrong durante la missione è stata: "Questo è un piccolo passo per un uomo, un grande balzo per l'umanità". Questa frase è diventata un simbolo della missione Apollo 11 e della conquista spaziale dell'uomo.

Armstrong era un astronauta ed esploratore spaziale statunitense che aveva una lunga carriera nella NASA prima di diventare comandante della missione Apollo 11. Prima di diventare astronauta, Armstrong era stato un aviatore e pilota collaudatore di velivoli sperimentali ad alta quota.

Dopo la missione Apollo 11, Armstrong s